In [1]:
from convnn_fused import FusedPrimeConv

Using /home/mkang2/.cache/torch_extensions/py311_cu121 as PyTorch extensions root...
Creating extension directory /home/mkang2/.cache/torch_extensions/py311_cu121/convnn_cuda...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/mkang2/.cache/torch_extensions/py311_cu121/convnn_cuda/build.ninja...
/mnt/local/python3.11.8/lib/python3.11/site-packages/torch/utils/cpp_extension.py:1967: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module convnn_cuda...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/3] c++ -MMD -MF convnn_binding.o.d -DTORCH_EXTENSION_NAME=convnn_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /mnt/local/python3.11.8/lib/python3.11/site-packages/torch/include -isystem /mnt/local/python3.11.8/lib/python3.11/site-packages/torch/include/torch/csrc/api/include -isystem /mnt/local/python3.11.8/lib/python3.11/site-packages/torch/include/TH -isystem /mnt/local/python3.11.8/lib/python3.11/site-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /mnt/local/python3.11.8/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /mnt/research/j.farias/mkang2/Convolutional-Nearest-Neighbor-Attention/CUDA/convnn_binding.cpp -o convnn_binding.o 
[2/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output convnn_kernel.cuda.o.d -DTORCH_EXTENSION_NAME=convnn_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc

Loading extension module convnn_cuda...


In [7]:
import torch 
import torch.nn as nn 
import torch.nn.functional as f
import numpy as np
"""Regular ConvNN Attention Implementation"""
class MultiHeadConvNNAttention(nn.Module):
    def __init__(self, 
                 d_hidden,
                 num_heads, 
                 attention_dropout, 
                 K, 
                 convolution_type='depthwise',
                 seq_length=197):

        super(MultiHeadConvNNAttention, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"

        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.attention_dropout = attention_dropout 
        self.d_k = d_hidden // num_heads 
        self.K = K 
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        self.in_channels = d_hidden // num_heads
        self.out_channels = d_hidden // num_heads

        if convolution_type == 'standard': 
            self.conv = nn.Conv1d(
                in_channels=self.in_channels,
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                bias=False
            )
        elif convolution_type == 'depthwise':
            self.conv = nn.Conv1d(
                in_channels=self.in_channels, 
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                groups=self.in_channels, 
                bias=False
            )
        elif convolution_type == 'depthwise-separable':
            self.conv = nn.Sequential(
                # Depthwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.in_channels,
                    kernel_size=self.K,
                    stride=self.K,
                    padding=0,
                    groups=self.in_channels,
                    bias=False
                ), 
                # Pointwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.out_channels,
                    kernel_size=1,
                    stride=1,
                    padding=0, 
                    bias=False
                )
            )
        self.conv.weight.data.fill_(1.0)

        self.fused_prime_conv = FusedPrimeConv(d_k = self.d_k, K = self.K)

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size() 
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) # (B, num_heads, seq_length, d_k)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size() 
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden)

    def _prime(self, v, qk, K):
        # v: (B*num_heads, d_k, seq_length), qk: (B*num_heads, seq_length, seq_length)
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K, dim=2, largest=True)

        topk_values = torch.softmax(topk_values, dim=-1)
        topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)

        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime

    def forward(self, x):
        B = x.shape[0]

        # Linear Projection + Split Heads 
        q = self.split_head(self.W_q(x)) # (B, NH, SL, DK)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        # Attention Matrix: (B, NH, SL, SL) - Q @ K^T
        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        # Merge B and num_heads into dim for prime & conv 
        ## (B, NH, SL, DK) → (B*NH, DK, SL) for v and (B, NH, SL, SL) → (B*NH, SL, SL)
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.seq_length)

        # Prime and Convolution 
        # prime = self._prime(v_merged, am_merged, self.K)
        # out = self.conv(prime) # (B*num_heads, d_k, seq_length) 

        out = self.fused_prime_conv(v_merged, am_merged)

        

        # Reshape back: (B*NH, DK, SL) → (B, NH, SL, DK)
        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out) 

        # Combine Heads and Final Linear Projection
        output = self.W_o(self.combine_heads(out)) # (B, SL, d_hidden)
        return output

In [12]:
import torch 
import torch.nn as nn 
import torch.nn.functional as f
import numpy as np
"""Regular ConvNN Attention Implementation"""
class MultiHeadConvNNAttention_Regular(nn.Module):
    def __init__(self, 
                 d_hidden,
                 num_heads, 
                 attention_dropout, 
                 K, 
                 convolution_type='depthwise',
                 seq_length=197):

        super(MultiHeadConvNNAttention_Regular, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"

        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.attention_dropout = attention_dropout 
        self.d_k = d_hidden // num_heads 
        self.K = K 
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        self.in_channels = d_hidden // num_heads
        self.out_channels = d_hidden // num_heads

        if convolution_type == 'standard': 
            self.conv = nn.Conv1d(
                in_channels=self.in_channels,
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                bias=False
            )
        elif convolution_type == 'depthwise':
            self.conv = nn.Conv1d(
                in_channels=self.in_channels, 
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                groups=self.in_channels, 
                bias=False
            )
        elif convolution_type == 'depthwise-separable':
            self.conv = nn.Sequential(
                # Depthwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.in_channels,
                    kernel_size=self.K,
                    stride=self.K,
                    padding=0,
                    groups=self.in_channels,
                    bias=False
                ), 
                # Pointwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.out_channels,
                    kernel_size=1,
                    stride=1,
                    padding=0, 
                    bias=False
                )
            )
        self.conv.weight.data.fill_(1.0)

        self.fused_prime_conv = FusedPrimeConv(d_k = self.d_k, K = self.K)

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size() 
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) # (B, num_heads, seq_length, d_k)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size() 
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden)

    def _prime(self, v, qk, K):
        # v: (B*num_heads, d_k, seq_length), qk: (B*num_heads, seq_length, seq_length)
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K, dim=2, largest=True)

        topk_values = torch.softmax(topk_values, dim=-1)
        topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)

        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime

    def forward(self, x):
        B = x.shape[0]

        # Linear Projection + Split Heads 
        q = self.split_head(self.W_q(x)) # (B, NH, SL, DK)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        # Attention Matrix: (B, NH, SL, SL) - Q @ K^T
        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        # Merge B and num_heads into dim for prime & conv 
        ## (B, NH, SL, DK) → (B*NH, DK, SL) for v and (B, NH, SL, SL) → (B*NH, SL, SL)
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.seq_length)

        # Prime and Convolution 
        prime = self._prime(v_merged, am_merged, self.K)
        out = self.conv(prime) # (B*num_heads, d_k, seq_length)        

        # Reshape back: (B*NH, DK, SL) → (B, NH, SL, DK)
        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out) 

        # Combine Heads and Final Linear Projection
        output = self.W_o(self.combine_heads(out)) # (B, SL, d_hidden)
        return output

In [14]:
convnn = MultiHeadConvNNAttention(768, 12, 0.0, 9).to('cuda')
convnn_regular = MultiHeadConvNNAttention_Regular(768, 12, 0.0, 9).to('cuda')
ex = torch.randn(3, 197, 768).to('cuda')
out = convnn(ex)
print(out.shape)

out_reg = convnn_regular(ex)
print(out_reg.shape)

torch.Size([3, 197, 768])
torch.Size([3, 197, 768])


## TESTING

In [15]:
"""
Test: Verify fused CUDA prime+conv produces identical output to original PyTorch implementation.

Both implementations use the SAME frozen weights (W_q, W_k, W_v, W_o, conv weights).
We compare at two levels:
  1. The prime+conv output in isolation (before reshape/output projection)
  2. The full attention module output (end-to-end)
"""
import torch
import torch.nn as nn
import numpy as np
import copy


# =============================================================================
# ORIGINAL IMPLEMENTATION (from your codebase)
# =============================================================================
class MultiHeadConvNNAttention(nn.Module):
    def __init__(self, d_hidden, num_heads, attention_dropout, K,
                 convolution_type='depthwise', seq_length=197):
        super(MultiHeadConvNNAttention, self).__init__()
        assert d_hidden % num_heads == 0

        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.attention_dropout = attention_dropout
        self.d_k = d_hidden // num_heads
        self.K = K
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        self.in_channels = d_hidden // num_heads
        self.out_channels = d_hidden // num_heads

        if convolution_type == 'depthwise':
            self.conv = nn.Conv1d(
                in_channels=self.in_channels,
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                groups=self.in_channels,
                bias=False
            )
        self.conv.weight.data.fill_(1.0)

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden)

    def _prime(self, v, qk, K):
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K, dim=2, largest=True)
        topk_values = torch.softmax(topk_values, dim=-1)
        topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)
        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime

    def forward(self, x):
        B = x.shape[0]

        q = self.split_head(self.W_q(x))
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.seq_length)

        prime = self._prime(v_merged, am_merged, self.K)
        out = self.conv(prime)

        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out)
        output = self.W_o(self.combine_heads(out))
        return output


# =============================================================================
# FUSED IMPLEMENTATION
# =============================================================================
class MultiHeadConvNNAttentionFused(nn.Module):
    """Same as original but uses the fused CUDA kernel for prime+conv."""

    def __init__(self, d_hidden, num_heads, attention_dropout, K,
                 seq_length=197):
        super(MultiHeadConvNNAttentionFused, self).__init__()
        assert d_hidden % num_heads == 0

        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.attention_dropout = attention_dropout
        self.d_k = d_hidden // num_heads
        self.K = K
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        # Fused prime+conv: conv weight is (d_k, K) instead of Conv1d
        from convnn_fused import FusedPrimeConv
        self.fused_prime_conv = FusedPrimeConv(d_k=self.d_k, K=self.K)

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden)

    def forward(self, x):
        B = x.shape[0]

        q = self.split_head(self.W_q(x))
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.seq_length)

        # === FUSED: replaces _prime() + self.conv() ===
        out = self.fused_prime_conv(v_merged, am_merged)

        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out)
        output = self.W_o(self.combine_heads(out))
        return output


# =============================================================================
# WEIGHT SYNCHRONIZATION
# =============================================================================
def sync_weights(original: MultiHeadConvNNAttention,
                 fused: MultiHeadConvNNAttentionFused):
    """
    Copy all weights from the original model to the fused model,
    ensuring the conv weights are mapped correctly.

    Original conv weight shape: (out_channels, 1, K) for depthwise Conv1d
      -> nn.Conv1d with groups=in_channels stores weights as (out_channels, 1, kernel_size)

    Fused conv weight shape: (d_k, K)
      -> simple 2D parameter, one row per channel
    """
    # Copy projection weights (identical structure)
    fused.W_q.weight.data.copy_(original.W_q.weight.data)
    fused.W_k.weight.data.copy_(original.W_k.weight.data)
    fused.W_v.weight.data.copy_(original.W_v.weight.data)
    fused.W_o.weight.data.copy_(original.W_o.weight.data)

    # Copy conv weights: (out_channels, 1, K) -> (d_k, K)
    # Squeeze out the group dimension (dim=1)
    original_conv_w = original.conv.weight.data.squeeze(1)  # (d_k, K)
    fused.fused_prime_conv.conv_weight.data.copy_(original_conv_w)


# =============================================================================
# TESTS
# =============================================================================
def test_isolated_prime_conv(device='cuda'):
    """
    Test 1: Compare prime+conv output in isolation.
    Feed identical (v_merged, am_merged) to both implementations.
    """
    print("=" * 70)
    print("TEST 1: Isolated prime+conv comparison")
    print("=" * 70)

    torch.manual_seed(42)

    B_NH = 48      # e.g., batch=4, num_heads=12
    d_k = 64
    SL = 197
    K = 8

    v = torch.randn(B_NH, d_k, SL, device=device)
    attn = torch.randn(B_NH, SL, SL, device=device)

    # --- Original PyTorch path ---
    # Top-K + softmax
    topk_values, topk_indices = torch.topk(attn, k=K, dim=2, largest=True)
    topk_values = torch.softmax(topk_values, dim=-1)

    b, c, t = v.shape
    topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
    topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)
    v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
    prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
    prime = topk_values_exp * prime
    prime = prime.view(b, c, -1)  # (B_NH, d_k, SL*K)

    # Depthwise Conv1d with stride=K, kernel=K (dot product per K-group)
    conv = nn.Conv1d(d_k, d_k, kernel_size=K, stride=K, padding=0,
                     groups=d_k, bias=False).to(device)
    conv.weight.data.fill_(1.0)  # match default init
    pytorch_out = conv(prime)  # (B_NH, d_k, SL)

    # --- Fused CUDA path ---
    from convnn_fused import FusedPrimeConv
    fused = FusedPrimeConv(d_k=d_k, K=K).to(device)
    # Sync conv weights: Conv1d weight is (d_k, 1, K), fused is (d_k, K)
    fused.conv_weight.data.copy_(conv.weight.data.squeeze(1))
    fused_out = fused(v, attn)

    # --- Compare ---
    max_diff = (pytorch_out - fused_out).abs().max().item()
    mean_diff = (pytorch_out - fused_out).abs().mean().item()
    rel_err = (pytorch_out - fused_out).abs() / (pytorch_out.abs() + 1e-8)
    max_rel = rel_err.max().item()

    print(f"  Shape:              {pytorch_out.shape}")
    print(f"  Max absolute diff:  {max_diff:.2e}")
    print(f"  Mean absolute diff: {mean_diff:.2e}")
    print(f"  Max relative error: {max_rel:.2e}")
    print(f"  Output range:       [{pytorch_out.min().item():.4f}, {pytorch_out.max().item():.4f}]")

    passed = max_diff < 1e-4
    print(f"  RESULT: {'PASS' if passed else 'FAIL'}")
    print()
    return passed


def test_full_module(device='cuda'):
    """
    Test 2: Full end-to-end module comparison.
    Same input, same frozen weights, compare final output.
    """
    print("=" * 70)
    print("TEST 2: Full module end-to-end comparison")
    print("=" * 70)

    torch.manual_seed(123)

    d_hidden = 768
    num_heads = 12
    K = 8
    seq_length = 197
    batch_size = 4

    # Create both modules
    original = MultiHeadConvNNAttention(
        d_hidden=d_hidden, num_heads=num_heads,
        attention_dropout=0.0,  # disable dropout for determinism
        K=K, seq_length=seq_length
    ).to(device)

    fused = MultiHeadConvNNAttentionFused(
        d_hidden=d_hidden, num_heads=num_heads,
        attention_dropout=0.0,
        K=K, seq_length=seq_length
    ).to(device)

    # Freeze both and sync weights
    original.eval()
    fused.eval()

    with torch.no_grad():
        sync_weights(original, fused)

    # Verify weight sync
    print("  Weight sync verification:")
    print(f"    W_q match: {torch.equal(original.W_q.weight, fused.W_q.weight)}")
    print(f"    W_k match: {torch.equal(original.W_k.weight, fused.W_k.weight)}")
    print(f"    W_v match: {torch.equal(original.W_v.weight, fused.W_v.weight)}")
    print(f"    W_o match: {torch.equal(original.W_o.weight, fused.W_o.weight)}")
    conv_w_orig = original.conv.weight.data.squeeze(1)
    conv_w_fused = fused.fused_prime_conv.conv_weight.data
    print(f"    Conv match: {torch.equal(conv_w_orig, conv_w_fused)}")
    print()

    # Run forward pass
    x = torch.randn(batch_size, seq_length, d_hidden, device=device)

    with torch.no_grad():
        out_original = original(x)
        out_fused = fused(x)

    # Compare
    max_diff = (out_original - out_fused).abs().max().item()
    mean_diff = (out_original - out_fused).abs().mean().item()
    rel_err = (out_original - out_fused).abs() / (out_original.abs() + 1e-8)
    max_rel = rel_err.max().item()

    print(f"  Output shape:       {out_original.shape}")
    print(f"  Max absolute diff:  {max_diff:.2e}")
    print(f"  Mean absolute diff: {mean_diff:.2e}")
    print(f"  Max relative error: {max_rel:.2e}")
    print(f"  Output range:       [{out_original.min().item():.4f}, {out_original.max().item():.4f}]")

    passed = max_diff < 1e-3  # slightly relaxed for full pipeline float accumulation
    print(f"  RESULT: {'PASS' if passed else 'FAIL'}")
    print()
    return passed


def test_multiple_configs(device='cuda'):
    """
    Test 3: Sweep over different K values and batch sizes.
    """
    print("=" * 70)
    print("TEST 3: Multiple configurations")
    print("=" * 70)

    configs = [
        {'batch': 1,  'd_hidden': 192, 'heads': 3,  'K': 4,  'sl': 50},
        {'batch': 2,  'd_hidden': 384, 'heads': 6,  'K': 8,  'sl': 100},
        {'batch': 4,  'd_hidden': 768, 'heads': 12, 'K': 16, 'sl': 197},
        {'batch': 8,  'd_hidden': 768, 'heads': 12, 'K': 32, 'sl': 197},
        {'batch': 1,  'd_hidden': 768, 'heads': 12, 'K': 64, 'sl': 197},
    ]

    all_passed = True
    for i, cfg in enumerate(configs):
        torch.manual_seed(i * 100)

        original = MultiHeadConvNNAttention(
            d_hidden=cfg['d_hidden'], num_heads=cfg['heads'],
            attention_dropout=0.0, K=cfg['K'], seq_length=cfg['sl']
        ).to(device)

        fused = MultiHeadConvNNAttentionFused(
            d_hidden=cfg['d_hidden'], num_heads=cfg['heads'],
            attention_dropout=0.0, K=cfg['K'], seq_length=cfg['sl']
        ).to(device)

        original.eval()
        fused.eval()

        with torch.no_grad():
            sync_weights(original, fused)
            x = torch.randn(cfg['batch'], cfg['sl'], cfg['d_hidden'], device=device)
            out_orig = original(x)
            out_fused = fused(x)

        max_diff = (out_orig - out_fused).abs().max().item()
        passed = max_diff < 1e-3

        status = "PASS" if passed else "FAIL"
        print(f"  Config {i+1}: B={cfg['batch']}, d={cfg['d_hidden']}, "
              f"H={cfg['heads']}, K={cfg['K']}, SL={cfg['sl']} "
              f"-> max_diff={max_diff:.2e} [{status}]")

        if not passed:
            all_passed = False

    print(f"\n  OVERALL: {'ALL PASSED' if all_passed else 'SOME FAILED'}")
    print()
    return all_passed


def test_gradient_flow(device='cuda'):
    """
    Test 4: Verify gradients flow correctly through the fused module.
    Compare gradients of W_q, W_k, W_v, W_o, and conv weights.
    """
    print("=" * 70)
    print("TEST 4: Gradient comparison")
    print("=" * 70)

    torch.manual_seed(77)

    d_hidden = 192
    num_heads = 3
    K = 4
    seq_length = 50
    batch_size = 2

    original = MultiHeadConvNNAttention(
        d_hidden=d_hidden, num_heads=num_heads,
        attention_dropout=0.0, K=K, seq_length=seq_length
    ).to(device)

    fused = MultiHeadConvNNAttentionFused(
        d_hidden=d_hidden, num_heads=num_heads,
        attention_dropout=0.0, K=K, seq_length=seq_length
    ).to(device)

    # Sync weights (not frozen this time — we need gradients)
    with torch.no_grad():
        sync_weights(original, fused)

    # Same input for both
    x = torch.randn(batch_size, seq_length, d_hidden, device=device)

    # Forward + backward for original
    out_orig = original(x)
    loss_orig = out_orig.sum()
    loss_orig.backward()

    # Forward + backward for fused
    out_fused = fused(x)
    loss_fused = out_fused.sum()
    loss_fused.backward()

    # Compare gradients
    grad_pairs = [
        ("W_q", original.W_q.weight.grad, fused.W_q.weight.grad),
        ("W_k", original.W_k.weight.grad, fused.W_k.weight.grad),
        ("W_v", original.W_v.weight.grad, fused.W_v.weight.grad),
        ("W_o", original.W_o.weight.grad, fused.W_o.weight.grad),
        ("conv_w",
         original.conv.weight.grad.squeeze(1),  # (d_k, 1, K) -> (d_k, K)
         fused.fused_prime_conv.conv_weight.grad),
    ]

    all_passed = True
    for name, g_orig, g_fused in grad_pairs:
        if g_orig is None or g_fused is None:
            print(f"  {name:8s}: gradient is None!")
            all_passed = False
            continue

        max_diff = (g_orig - g_fused).abs().max().item()
        mean_diff = (g_orig - g_fused).abs().mean().item()
        passed = max_diff < 1e-3

        status = "PASS" if passed else "FAIL"
        print(f"  {name:8s}: max_diff={max_diff:.2e}, mean_diff={mean_diff:.2e} [{status}]")

        if not passed:
            all_passed = False

    print(f"\n  OVERALL: {'ALL PASSED' if all_passed else 'SOME FAILED'}")
    print()
    return all_passed


# =============================================================================
# MAIN
# =============================================================================
if __name__ == '__main__':
    assert torch.cuda.is_available(), "CUDA is required for this test"
    device = 'cuda'

    results = []
    results.append(("Isolated prime+conv", test_isolated_prime_conv(device)))
    results.append(("Full module e2e",     test_full_module(device)))
    results.append(("Multiple configs",    test_multiple_configs(device)))
    results.append(("Gradient flow",       test_gradient_flow(device)))

    print("=" * 70)
    print("SUMMARY")
    print("=" * 70)
    all_good = True
    for name, passed in results:
        status = "PASS" if passed else "FAIL"
        print(f"  {name:30s} [{status}]")
        if not passed:
            all_good = False

    print()
    if all_good:
        print("All tests passed! Fused kernel matches original implementation.")
    else:
        print("Some tests FAILED. Check output above for details.")

TEST 1: Isolated prime+conv comparison
  Shape:              torch.Size([48, 64, 197])
  Max absolute diff:  4.77e-07
  Mean absolute diff: 1.53e-08
  Max relative error: 1.65e-02
  Output range:       [-2.0229, 2.1652]
  RESULT: PASS

TEST 2: Full module end-to-end comparison
  Weight sync verification:
    W_q match: True
    W_k match: True
    W_v match: True
    W_o match: True
    Conv match: True

  Output shape:       torch.Size([4, 197, 768])
  Max absolute diff:  1.34e-07
  Mean absolute diff: 1.54e-08
  Max relative error: 1.00e-01
  Output range:       [-0.5740, 0.5795]
  RESULT: PASS

TEST 3: Multiple configurations
  Config 1: B=1, d=192, H=3, K=4, SL=50 -> max_diff=8.94e-08 [PASS]
  Config 2: B=2, d=384, H=6, K=8, SL=100 -> max_diff=1.19e-07 [PASS]
  Config 3: B=4, d=768, H=12, K=16, SL=197 -> max_diff=1.19e-07 [PASS]
  Config 4: B=8, d=768, H=12, K=32, SL=197 -> max_diff=1.79e-07 [PASS]
  Config 5: B=1, d=768, H=12, K=64, SL=197 -> max_diff=6.71e-08 [PASS]

  OVERALL: A

## Speed Testing

In [17]:
"""
Speed Benchmark: Original PyTorch ConvNN Attention vs Fused CUDA Kernel

Measures forward and backward pass times across multiple configurations.
Uses proper CUDA timing (events + synchronization) and warmup iterations.
"""
import torch
import torch.nn as nn
import numpy as np
import time
import csv
import os


# =============================================================================
# ORIGINAL IMPLEMENTATION
# =============================================================================
class MultiHeadConvNNAttention(nn.Module):
    def __init__(self, d_hidden, num_heads, attention_dropout, K,
                 convolution_type='depthwise', seq_length=197):
        super().__init__()
        assert d_hidden % num_heads == 0

        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.d_k = d_hidden // num_heads
        self.K = K
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        self.conv = nn.Conv1d(
            in_channels=self.d_k, out_channels=self.d_k,
            kernel_size=K, stride=K, padding=0,
            groups=self.d_k, bias=False
        )
        self.conv.weight.data.fill_(1.0)

    def split_head(self, x):
        B, S, _ = x.size()
        return x.view(B, S, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        B, _, S, _ = x.size()
        return x.transpose(1, 2).contiguous().view(B, S, self.d_hidden)

    def _prime(self, v, qk, K):
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K, dim=2, largest=True)
        topk_values = torch.softmax(topk_values, dim=-1)
        topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)
        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime

    def forward(self, x):
        B = x.shape[0]
        q = self.split_head(self.W_q(x))
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.seq_length)

        prime = self._prime(v_merged, am_merged, self.K)
        out = self.conv(prime)

        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out)
        return self.W_o(self.combine_heads(out))


# =============================================================================
# FUSED IMPLEMENTATION
# =============================================================================
class MultiHeadConvNNAttentionFused(nn.Module):
    def __init__(self, d_hidden, num_heads, attention_dropout, K, seq_length=197):
        super().__init__()
        assert d_hidden % num_heads == 0

        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.d_k = d_hidden // num_heads
        self.K = K
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        from convnn_fused import FusedPrimeConv
        self.fused_prime_conv = FusedPrimeConv(d_k=self.d_k, K=self.K)

    def split_head(self, x):
        B, S, _ = x.size()
        return x.view(B, S, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        B, _, S, _ = x.size()
        return x.transpose(1, 2).contiguous().view(B, S, self.d_hidden)

    def forward(self, x):
        B = x.shape[0]
        q = self.split_head(self.W_q(x))
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.seq_length)

        out = self.fused_prime_conv(v_merged, am_merged)

        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out)
        return self.W_o(self.combine_heads(out))


# =============================================================================
# WEIGHT SYNC
# =============================================================================
def sync_weights(original, fused):
    with torch.no_grad():
        fused.W_q.weight.data.copy_(original.W_q.weight.data)
        fused.W_k.weight.data.copy_(original.W_k.weight.data)
        fused.W_v.weight.data.copy_(original.W_v.weight.data)
        fused.W_o.weight.data.copy_(original.W_o.weight.data)
        fused.fused_prime_conv.conv_weight.data.copy_(
            original.conv.weight.data.squeeze(1)
        )


# =============================================================================
# CUDA-ACCURATE TIMING UTILITY
# =============================================================================
def benchmark_fn(fn, warmup=20, repeats=100):
    """
    Time a function using CUDA events for accurate GPU timing.

    Returns:
        dict with 'mean_ms', 'std_ms', 'min_ms', 'max_ms', 'median_ms'
    """
    # Warmup: let CUDA JIT, caching, etc. stabilize
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    # Timed runs using CUDA events (not wall-clock time)
    timings = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

        start.record()
        fn()
        end.record()

        torch.cuda.synchronize()
        timings.append(start.elapsed_time(end))  # milliseconds

    timings = np.array(timings)
    return {
        'mean_ms': timings.mean(),
        'std_ms': timings.std(),
        'min_ms': timings.min(),
        'max_ms': timings.max(),
        'median_ms': np.median(timings),
    }


# =============================================================================
# BENCHMARK SUITE
# =============================================================================
def run_benchmark(config, device='cuda', warmup=20, repeats=100):
    """
    Benchmark a single configuration. Returns timing results for
    forward-only and forward+backward for both implementations.
    """
    torch.manual_seed(0)

    B = config['batch']
    d_hidden = config['d_hidden']
    num_heads = config['heads']
    K = config['K']
    SL = config['sl']

    # Build models
    original = MultiHeadConvNNAttention(
        d_hidden=d_hidden, num_heads=num_heads,
        attention_dropout=0.0, K=K, seq_length=SL
    ).to(device)

    fused = MultiHeadConvNNAttentionFused(
        d_hidden=d_hidden, num_heads=num_heads,
        attention_dropout=0.0, K=K, seq_length=SL
    ).to(device)

    sync_weights(original, fused)

    original.train()
    fused.train()

    x = torch.randn(B, SL, d_hidden, device=device)

    # --- Forward-only benchmarks ---
    def fwd_original():
        with torch.no_grad():
            original(x)

    def fwd_fused():
        with torch.no_grad():
            fused(x)

    t_fwd_orig = benchmark_fn(fwd_original, warmup=warmup, repeats=repeats)
    t_fwd_fused = benchmark_fn(fwd_fused, warmup=warmup, repeats=repeats)

    # --- Forward + Backward benchmarks ---
    def fwd_bwd_original():
        # Zero grads
        original.zero_grad(set_to_none=True)
        out = original(x)
        loss = out.sum()
        loss.backward()

    def fwd_bwd_fused():
        fused.zero_grad(set_to_none=True)
        out = fused(x)
        loss = out.sum()
        loss.backward()

    t_fwdbwd_orig = benchmark_fn(fwd_bwd_original, warmup=warmup, repeats=repeats)
    t_fwdbwd_fused = benchmark_fn(fwd_bwd_fused, warmup=warmup, repeats=repeats)

    return {
        'config': config,
        'fwd_original': t_fwd_orig,
        'fwd_fused': t_fwd_fused,
        'fwdbwd_original': t_fwdbwd_orig,
        'fwdbwd_fused': t_fwdbwd_fused,
    }


def format_timing(t):
    """Format timing dict as 'mean ± std ms'."""
    return f"{t['mean_ms']:8.3f} ± {t['std_ms']:.3f} ms"


def print_results(all_results):
    """Pretty-print benchmark results as a table."""

    # Header
    print()
    print("=" * 130)
    print(f"{'Config':>40s}  |  {'Forward (ms)':^45s}  |  {'Forward+Backward (ms)':^45s}")
    print(f"{'':>40s}  |  {'Original':>20s}  {'Fused':>20s}  |  {'Original':>20s}  {'Fused':>20s}")
    print("-" * 130)

    for r in all_results:
        cfg = r['config']
        label = f"B={cfg['batch']}, d={cfg['d_hidden']}, H={cfg['heads']}, K={cfg['K']}, SL={cfg['sl']}"

        fo = r['fwd_original']
        ff = r['fwd_fused']
        bo = r['fwdbwd_original']
        bf = r['fwdbwd_fused']

        print(f"{label:>40s}  |  {format_timing(fo):>20s}  {format_timing(ff):>20s}  "
              f"|  {format_timing(bo):>20s}  {format_timing(bf):>20s}")

    # Speedup summary
    print()
    print("=" * 130)
    print(f"{'Config':>40s}  |  {'Fwd Speedup':>12s}  {'Fwd+Bwd Speedup':>16s}  "
          f"{'Fwd ΔGPU mem':>14s}  {'Fwd+Bwd ΔGPU mem':>18s}")
    print("-" * 130)

    for r in all_results:
        cfg = r['config']
        label = f"B={cfg['batch']}, d={cfg['d_hidden']}, H={cfg['heads']}, K={cfg['K']}, SL={cfg['sl']}"

        fwd_speedup = r['fwd_original']['mean_ms'] / r['fwd_fused']['mean_ms']
        bwd_speedup = r['fwdbwd_original']['mean_ms'] / r['fwdbwd_fused']['mean_ms']

        print(f"{label:>40s}  |  {fwd_speedup:>11.2f}x  {bwd_speedup:>15.2f}x")

    print("=" * 130)


def run_memory_comparison(config, device='cuda'):
    """Measure peak GPU memory for both implementations."""
    torch.manual_seed(0)

    B = config['batch']
    d_hidden = config['d_hidden']
    num_heads = config['heads']
    K = config['K']
    SL = config['sl']

    results = {}

    for name, ModelClass in [('original', MultiHeadConvNNAttention),
                              ('fused', MultiHeadConvNNAttentionFused)]:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        if name == 'original':
            model = ModelClass(
                d_hidden=d_hidden, num_heads=num_heads,
                attention_dropout=0.0, K=K, seq_length=SL
            ).to(device)
        else:
            model = ModelClass(
                d_hidden=d_hidden, num_heads=num_heads,
                attention_dropout=0.0, K=K, seq_length=SL
            ).to(device)

        x = torch.randn(B, SL, d_hidden, device=device)

        # Forward + backward to capture peak memory
        torch.cuda.reset_peak_memory_stats()
        model.zero_grad(set_to_none=True)
        out = model(x)
        loss = out.sum()
        loss.backward()
        torch.cuda.synchronize()

        results[name] = {
            'peak_mb': torch.cuda.max_memory_allocated() / (1024 ** 2),
            'current_mb': torch.cuda.memory_allocated() / (1024 ** 2),
        }

        del model, x, out, loss
        torch.cuda.empty_cache()

    return results


def save_csv(all_results, filepath='benchmark_results.csv'):
    """Save benchmark results to CSV for plotting."""
    rows = []
    for r in all_results:
        cfg = r['config']
        row = {
            'batch': cfg['batch'],
            'd_hidden': cfg['d_hidden'],
            'heads': cfg['heads'],
            'K': cfg['K'],
            'seq_length': cfg['sl'],
            'fwd_orig_mean_ms': r['fwd_original']['mean_ms'],
            'fwd_orig_std_ms': r['fwd_original']['std_ms'],
            'fwd_fused_mean_ms': r['fwd_fused']['mean_ms'],
            'fwd_fused_std_ms': r['fwd_fused']['std_ms'],
            'fwdbwd_orig_mean_ms': r['fwdbwd_original']['mean_ms'],
            'fwdbwd_orig_std_ms': r['fwdbwd_original']['std_ms'],
            'fwdbwd_fused_mean_ms': r['fwdbwd_fused']['mean_ms'],
            'fwdbwd_fused_std_ms': r['fwdbwd_fused']['std_ms'],
            'fwd_speedup': r['fwd_original']['mean_ms'] / r['fwd_fused']['mean_ms'],
            'fwdbwd_speedup': r['fwdbwd_original']['mean_ms'] / r['fwdbwd_fused']['mean_ms'],
        }
        rows.append(row)

    with open(filepath, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nResults saved to {filepath}")


# =============================================================================
# MAIN
# =============================================================================
if __name__ == '__main__':
    assert torch.cuda.is_available(), "CUDA required"
    device = 'cuda'

    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA: {torch.version.cuda}")
    print()

    # ---- Configurations ----
    # Covers your actual ViT-Base setup + scaling experiments
    configs = [
        # Small (fast sanity check)
        {'batch': 4,  'd_hidden': 192,  'heads': 3,   'K': 8,   'sl': 50,  'label': 'Small'},
        # ViT-Base (your primary setup)
        {'batch': 4,  'd_hidden': 768,  'heads': 12,  'K': 8,   'sl': 197, 'label': 'ViT-B (B=4)'},
        {'batch': 16, 'd_hidden': 768,  'heads': 12,  'K': 8,   'sl': 197, 'label': 'ViT-B (B=16)'},
        {'batch': 64, 'd_hidden': 768,  'heads': 12,  'K': 8,   'sl': 197, 'label': 'ViT-B (B=64)'},
        # Vary K (nearest neighbors)
        {'batch': 16, 'd_hidden': 768,  'heads': 12,  'K': 4,   'sl': 197, 'label': 'ViT-B K=4'},
        {'batch': 16, 'd_hidden': 768,  'heads': 12,  'K': 16,  'sl': 197, 'label': 'ViT-B K=16'},
        {'batch': 16, 'd_hidden': 768,  'heads': 12,  'K': 32,  'sl': 197, 'label': 'ViT-B K=32'},
        {'batch': 16, 'd_hidden': 768,  'heads': 12,  'K': 64,  'sl': 197, 'label': 'ViT-B K=64'},
        # ViT-Large
        {'batch': 4,  'd_hidden': 1024, 'heads': 16,  'K': 8,   'sl': 197, 'label': 'ViT-L (B=4)'},
        {'batch': 16, 'd_hidden': 1024, 'heads': 16,  'K': 8,   'sl': 197, 'label': 'ViT-L (B=16)'},
        # Dense prediction (longer sequences for detection/segmentation)
        {'batch': 4,  'd_hidden': 768,  'heads': 12,  'K': 8,   'sl': 576, 'label': 'ViT-B SL=576'},
        {'batch': 4,  'd_hidden': 768,  'heads': 12,  'K': 8,   'sl': 1024,'label': 'ViT-B SL=1024'},
    ]

    # ---- Run benchmarks ----
    print("Running speed benchmarks (this may take a few minutes)...")
    print()

    all_results = []
    for i, cfg in enumerate(configs):
        label = cfg.pop('label', '')
        print(f"[{i+1}/{len(configs)}] {label}: B={cfg['batch']}, d={cfg['d_hidden']}, "
              f"H={cfg['heads']}, K={cfg['K']}, SL={cfg['sl']}...", flush=True)
        try:
            result = run_benchmark(cfg, device=device, warmup=20, repeats=100)
            all_results.append(result)

            # Quick inline preview
            fwd_sp = result['fwd_original']['mean_ms'] / result['fwd_fused']['mean_ms']
            bwd_sp = result['fwdbwd_original']['mean_ms'] / result['fwdbwd_fused']['mean_ms']
            print(f"         -> Fwd: {fwd_sp:.2f}x, Fwd+Bwd: {bwd_sp:.2f}x")
        except RuntimeError as e:
            print(f"         -> SKIPPED (OOM or error): {e}")

        # Clear between configs
        torch.cuda.empty_cache()

    # ---- Print results ----
    print_results(all_results)

    # ---- Memory comparison (ViT-Base default) ----
    print("\n\nMemory Usage Comparison (ViT-Base, B=16, K=8):")
    print("-" * 50)
    mem_cfg = {'batch': 16, 'd_hidden': 768, 'heads': 12, 'K': 8, 'sl': 197}
    mem = run_memory_comparison(mem_cfg, device=device)
    for name, stats in mem.items():
        print(f"  {name:10s}: peak={stats['peak_mb']:.1f} MB")
    if mem['original']['peak_mb'] > 0:
        savings = (1 - mem['fused']['peak_mb'] / mem['original']['peak_mb']) * 100
        print(f"  Memory savings: {savings:.1f}%")

    # ---- Save CSV ----
    save_csv(all_results, 'benchmark_results.csv')

GPU: NVIDIA A100 80GB PCIe (79.3 GB)
PyTorch: 2.3.1+cu121
CUDA: 12.1

Running speed benchmarks (this may take a few minutes)...

[1/12] Small: B=4, d=192, H=3, K=8, SL=50...
         -> Fwd: 1.08x, Fwd+Bwd: 1.12x
[2/12] ViT-B (B=4): B=4, d=768, H=12, K=8, SL=197...
         -> Fwd: 0.23x, Fwd+Bwd: 0.24x
[3/12] ViT-B (B=16): B=16, d=768, H=12, K=8, SL=197...
         -> Fwd: 0.17x, Fwd+Bwd: 0.14x
[4/12] ViT-B (B=64): B=64, d=768, H=12, K=8, SL=197...
         -> Fwd: 0.15x, Fwd+Bwd: 0.08x
[5/12] ViT-B K=4: B=16, d=768, H=12, K=4, SL=197...
         -> Fwd: 0.32x, Fwd+Bwd: 0.19x
[6/12] ViT-B K=16: B=16, d=768, H=12, K=16, SL=197...
         -> Fwd: 0.06x, Fwd+Bwd: 0.10x
[7/12] ViT-B K=32: B=16, d=768, H=12, K=32, SL=197...
         -> Fwd: 0.02x, Fwd+Bwd: 0.08x
[8/12] ViT-B K=64: B=16, d=768, H=12, K=64, SL=197...
         -> Fwd: 0.02x, Fwd+Bwd: 0.10x
[9/12] ViT-L (B=4): B=4, d=1024, H=16, K=8, SL=197...
         -> Fwd: 0.23x, Fwd+Bwd: 0.23x
[10/12] ViT-L (B=16): B=16, d=1024, H=16, K=

## Triton Implementation

In [18]:
import triton
import triton.language as tl

@triton.jit
def prime_conv_kernel(
    v_ptr, attn_ptr, conv_w_ptr, output_ptr,
    SL: tl.constexpr, K: tl.constexpr, d_k: tl.constexpr,
):
    # Each program instance handles one (batch_head, channel, output_pos)
    b = tl.program_id(0)      # batch * num_heads
    c = tl.program_id(1)      # channel index
    t_out = tl.program_id(2)  # output sequence position

    # Load attention row: attn[b, t_out, :]
    attn_offsets = b * SL * SL + t_out * SL + tl.arange(0, SL)
    attn_row = tl.load(attn_ptr + attn_offsets)  # (SL,)

    # Top-K: Triton doesn't have a native topk, so we use iterative masking
    # For each of K iterations, find the max, record it, mask it out
    topk_vals = tl.zeros([K], dtype=tl.float32)
    topk_idxs = tl.zeros([K], dtype=tl.int32)
    masked_row = attn_row

    for i in range(K):
        max_val = tl.max(masked_row, axis=0)
        max_idx = tl.argmax(masked_row, axis=0)
        # Store
        tl.store(topk_vals + i, max_val)  # conceptual
        tl.store(topk_idxs + i, max_idx)
        # Mask out the found max
        mask = tl.arange(0, SL) == max_idx
        masked_row = tl.where(mask, float('-inf'), masked_row)

    # Softmax over top-K values
    topk_max = tl.max(topk_vals, axis=0)
    topk_exp = tl.exp(topk_vals - topk_max)
    topk_softmax = topk_exp / tl.sum(topk_exp, axis=0)

    # Gather from v, weight, and convolve — all fused
    result = 0.0
    for i in range(K):
        v_offset = b * d_k * SL + c * SL + topk_idxs[i]
        v_val = tl.load(v_ptr + v_offset)
        w_val = tl.load(conv_w_ptr + c * K + i)
        result += topk_softmax[i] * v_val * w_val

    # Store output
    out_offset = b * d_k * SL + c * SL + t_out
    tl.store(output_ptr + out_offset, result)

In [19]:
def prime_conv_triton(v, attn, conv_w, K):
    B_NH, d_k, SL = v.shape
    output = torch.empty_like(v)

    grid = (B_NH, d_k, SL)
    prime_conv_kernel[grid](
        v, attn, conv_w, output,
        SL=SL, K=K, d_k=d_k,
    )
    return output

In [ ]:
"""
Test: Verify fused CUDA prime+conv produces identical output to original PyTorch implementation.

Both implementations use the SAME frozen weights (W_q, W_k, W_v, W_o, conv weights).
We compare at two levels:
  1. The prime+conv output in isolation (before reshape/output projection)
  2. The full attention module output (end-to-end)
"""
import torch
import torch.nn as nn
import numpy as np
import copy


# =============================================================================
# ORIGINAL IMPLEMENTATION (from your codebase)
# =============================================================================
class MultiHeadConvNNAttention(nn.Module):
    def __init__(self, d_hidden, num_heads, attention_dropout, K,
                 convolution_type='depthwise', seq_length=197):
        super(MultiHeadConvNNAttention, self).__init__()
        assert d_hidden % num_heads == 0

        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.attention_dropout = attention_dropout
        self.d_k = d_hidden // num_heads
        self.K = K
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        self.in_channels = d_hidden // num_heads
        self.out_channels = d_hidden // num_heads

        if convolution_type == 'depthwise':
            self.conv = nn.Conv1d(
                in_channels=self.in_channels,
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                groups=self.in_channels,
                bias=False
            )
        self.conv.weight.data.fill_(1.0)

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden)

    def _prime(self, v, qk, K):
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K, dim=2, largest=True)
        topk_values = torch.softmax(topk_values, dim=-1)
        topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)
        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime

    def forward(self, x):
        B = x.shape[0]

        q = self.split_head(self.W_q(x))
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.seq_length)

        prime = self._prime(v_merged, am_merged, self.K)
        out = self.conv(prime)

        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out)
        output = self.W_o(self.combine_heads(out))
        return output


# =============================================================================
# FUSED IMPLEMENTATION
# =============================================================================
class MultiHeadConvNNAttentionFused(nn.Module):
    """Same as original but uses the fused CUDA kernel for prime+conv."""

    def __init__(self, d_hidden, num_heads, attention_dropout, K,
                 seq_length=197):
        super(MultiHeadConvNNAttentionFused, self).__init__()
        assert d_hidden % num_heads == 0

        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.attention_dropout = attention_dropout
        self.d_k = d_hidden // num_heads
        self.K = K
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        # Fused prime+conv: conv weight is (d_k, K) instead of Conv1d
        from convnn_fused import FusedPrimeConv
        self.fused_prime_conv = FusedPrimeConv(d_k=self.d_k, K=self.K)

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden)

    def forward(self, x):
        B = x.shape[0]

        q = self.split_head(self.W_q(x))
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.seq_length)

        # === FUSED: replaces _prime() + self.conv() ===
        out = self.fused_prime_conv(v_merged, am_merged)

        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out)
        output = self.W_o(self.combine_heads(out))
        return output


# =============================================================================
# WEIGHT SYNCHRONIZATION
# =============================================================================
def sync_weights(original: MultiHeadConvNNAttention,
                 fused: MultiHeadConvNNAttentionFused):
    """
    Copy all weights from the original model to the fused model,
    ensuring the conv weights are mapped correctly.

    Original conv weight shape: (out_channels, 1, K) for depthwise Conv1d
      -> nn.Conv1d with groups=in_channels stores weights as (out_channels, 1, kernel_size)

    Fused conv weight shape: (d_k, K)
      -> simple 2D parameter, one row per channel
    """
    # Copy projection weights (identical structure)
    fused.W_q.weight.data.copy_(original.W_q.weight.data)
    fused.W_k.weight.data.copy_(original.W_k.weight.data)
    fused.W_v.weight.data.copy_(original.W_v.weight.data)
    fused.W_o.weight.data.copy_(original.W_o.weight.data)

    # Copy conv weights: (out_channels, 1, K) -> (d_k, K)
    # Squeeze out the group dimension (dim=1)
    original_conv_w = original.conv.weight.data.squeeze(1)  # (d_k, K)
    fused.fused_prime_conv.conv_weight.data.copy_(original_conv_w)


# =============================================================================
# TESTS
# =============================================================================
def test_isolated_prime_conv(device='cuda'):
    """
    Test 1: Compare prime+conv output in isolation.
    Feed identical (v_merged, am_merged) to both implementations.
    """
    print("=" * 70)
    print("TEST 1: Isolated prime+conv comparison")
    print("=" * 70)

    torch.manual_seed(42)

    B_NH = 48      # e.g., batch=4, num_heads=12
    d_k = 64
    SL = 197
    K = 8

    v = torch.randn(B_NH, d_k, SL, device=device)
    attn = torch.randn(B_NH, SL, SL, device=device)

    # --- Original PyTorch path ---
    # Top-K + softmax
    topk_values, topk_indices = torch.topk(attn, k=K, dim=2, largest=True)
    topk_values = torch.softmax(topk_values, dim=-1)

    b, c, t = v.shape
    topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
    topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)
    v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
    prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
    prime = topk_values_exp * prime
    prime = prime.view(b, c, -1)  # (B_NH, d_k, SL*K)

    # Depthwise Conv1d with stride=K, kernel=K (dot product per K-group)
    conv = nn.Conv1d(d_k, d_k, kernel_size=K, stride=K, padding=0,
                     groups=d_k, bias=False).to(device)
    conv.weight.data.fill_(1.0)  # match default init
    pytorch_out = conv(prime)  # (B_NH, d_k, SL)

    # --- Fused CUDA path ---
    from convnn_fused import FusedPrimeConv
    fused = FusedPrimeConv(d_k=d_k, K=K).to(device)
    # Sync conv weights: Conv1d weight is (d_k, 1, K), fused is (d_k, K)
    fused.conv_weight.data.copy_(conv.weight.data.squeeze(1))
    fused_out = fused(v, attn)

    # --- Compare ---
    max_diff = (pytorch_out - fused_out).abs().max().item()
    mean_diff = (pytorch_out - fused_out).abs().mean().item()
    rel_err = (pytorch_out - fused_out).abs() / (pytorch_out.abs() + 1e-8)
    max_rel = rel_err.max().item()

    print(f"  Shape:              {pytorch_out.shape}")
    print(f"  Max absolute diff:  {max_diff:.2e}")
    print(f"  Mean absolute diff: {mean_diff:.2e}")
    print(f"  Max relative error: {max_rel:.2e}")
    print(f"  Output range:       [{pytorch_out.min().item():.4f}, {pytorch_out.max().item():.4f}]")

    passed = max_diff < 1e-4
    print(f"  RESULT: {'PASS' if passed else 'FAIL'}")
    print()
    return passed


def test_full_module(device='cuda'):
    """
    Test 2: Full end-to-end module comparison.
    Same input, same frozen weights, compare final output.
    """
    print("=" * 70)
    print("TEST 2: Full module end-to-end comparison")
    print("=" * 70)

    torch.manual_seed(123)

    d_hidden = 768
    num_heads = 12
    K = 8
    seq_length = 197
    batch_size = 4

    # Create both modules
    original = MultiHeadConvNNAttention(
        d_hidden=d_hidden, num_heads=num_heads,
        attention_dropout=0.0,  # disable dropout for determinism
        K=K, seq_length=seq_length
    ).to(device)

    fused = MultiHeadConvNNAttentionFused(
        d_hidden=d_hidden, num_heads=num_heads,
        attention_dropout=0.0,
        K=K, seq_length=seq_length
    ).to(device)

    # Freeze both and sync weights
    original.eval()
    fused.eval()

    with torch.no_grad():
        sync_weights(original, fused)

    # Verify weight sync
    print("  Weight sync verification:")
    print(f"    W_q match: {torch.equal(original.W_q.weight, fused.W_q.weight)}")
    print(f"    W_k match: {torch.equal(original.W_k.weight, fused.W_k.weight)}")
    print(f"    W_v match: {torch.equal(original.W_v.weight, fused.W_v.weight)}")
    print(f"    W_o match: {torch.equal(original.W_o.weight, fused.W_o.weight)}")
    conv_w_orig = original.conv.weight.data.squeeze(1)
    conv_w_fused = fused.fused_prime_conv.conv_weight.data
    print(f"    Conv match: {torch.equal(conv_w_orig, conv_w_fused)}")
    print()

    # Run forward pass
    x = torch.randn(batch_size, seq_length, d_hidden, device=device)

    with torch.no_grad():
        out_original = original(x)
        out_fused = fused(x)

    # Compare
    max_diff = (out_original - out_fused).abs().max().item()
    mean_diff = (out_original - out_fused).abs().mean().item()
    rel_err = (out_original - out_fused).abs() / (out_original.abs() + 1e-8)
    max_rel = rel_err.max().item()

    print(f"  Output shape:       {out_original.shape}")
    print(f"  Max absolute diff:  {max_diff:.2e}")
    print(f"  Mean absolute diff: {mean_diff:.2e}")
    print(f"  Max relative error: {max_rel:.2e}")
    print(f"  Output range:       [{out_original.min().item():.4f}, {out_original.max().item():.4f}]")

    passed = max_diff < 1e-3  # slightly relaxed for full pipeline float accumulation
    print(f"  RESULT: {'PASS' if passed else 'FAIL'}")
    print()
    return passed


def test_multiple_configs(device='cuda'):
    """
    Test 3: Sweep over different K values and batch sizes.
    """
    print("=" * 70)
    print("TEST 3: Multiple configurations")
    print("=" * 70)

    configs = [
        {'batch': 1,  'd_hidden': 192, 'heads': 3,  'K': 4,  'sl': 50},
        {'batch': 2,  'd_hidden': 384, 'heads': 6,  'K': 8,  'sl': 100},
        {'batch': 4,  'd_hidden': 768, 'heads': 12, 'K': 16, 'sl': 197},
        {'batch': 8,  'd_hidden': 768, 'heads': 12, 'K': 32, 'sl': 197},
        {'batch': 1,  'd_hidden': 768, 'heads': 12, 'K': 64, 'sl': 197},
    ]

    all_passed = True
    for i, cfg in enumerate(configs):
        torch.manual_seed(i * 100)

        original = MultiHeadConvNNAttention(
            d_hidden=cfg['d_hidden'], num_heads=cfg['heads'],
            attention_dropout=0.0, K=cfg['K'], seq_length=cfg['sl']
        ).to(device)

        fused = MultiHeadConvNNAttentionFused(
            d_hidden=cfg['d_hidden'], num_heads=cfg['heads'],
            attention_dropout=0.0, K=cfg['K'], seq_length=cfg['sl']
        ).to(device)

        original.eval()
        fused.eval()

        with torch.no_grad():
            sync_weights(original, fused)
            x = torch.randn(cfg['batch'], cfg['sl'], cfg['d_hidden'], device=device)
            out_orig = original(x)
            out_fused = fused(x)

        max_diff = (out_orig - out_fused).abs().max().item()
        passed = max_diff < 1e-3

        status = "PASS" if passed else "FAIL"
        print(f"  Config {i+1}: B={cfg['batch']}, d={cfg['d_hidden']}, "
              f"H={cfg['heads']}, K={cfg['K']}, SL={cfg['sl']} "
              f"-> max_diff={max_diff:.2e} [{status}]")

        if not passed:
            all_passed = False

    print(f"\n  OVERALL: {'ALL PASSED' if all_passed else 'SOME FAILED'}")
    print()
    return all_passed


def test_gradient_flow(device='cuda'):
    """
    Test 4: Verify gradients flow correctly through the fused module.
    Compare gradients of W_q, W_k, W_v, W_o, and conv weights.
    """
    print("=" * 70)
    print("TEST 4: Gradient comparison")
    print("=" * 70)

    torch.manual_seed(77)

    d_hidden = 192
    num_heads = 3
    K = 4
    seq_length = 50
    batch_size = 2

    original = MultiHeadConvNNAttention(
        d_hidden=d_hidden, num_heads=num_heads,
        attention_dropout=0.0, K=K, seq_length=seq_length
    ).to(device)

    fused = MultiHeadConvNNAttentionFused(
        d_hidden=d_hidden, num_heads=num_heads,
        attention_dropout=0.0, K=K, seq_length=seq_length
    ).to(device)

    # Sync weights (not frozen this time — we need gradients)
    with torch.no_grad():
        sync_weights(original, fused)

    # Same input for both
    x = torch.randn(batch_size, seq_length, d_hidden, device=device)

    # Forward + backward for original
    out_orig = original(x)
    loss_orig = out_orig.sum()
    loss_orig.backward()

    # Forward + backward for fused
    out_fused = fused(x)
    loss_fused = out_fused.sum()
    loss_fused.backward()

    # Compare gradients
    grad_pairs = [
        ("W_q", original.W_q.weight.grad, fused.W_q.weight.grad),
        ("W_k", original.W_k.weight.grad, fused.W_k.weight.grad),
        ("W_v", original.W_v.weight.grad, fused.W_v.weight.grad),
        ("W_o", original.W_o.weight.grad, fused.W_o.weight.grad),
        ("conv_w",
         original.conv.weight.grad.squeeze(1),  # (d_k, 1, K) -> (d_k, K)
         fused.fused_prime_conv.conv_weight.grad),
    ]

    all_passed = True
    for name, g_orig, g_fused in grad_pairs:
        if g_orig is None or g_fused is None:
            print(f"  {name:8s}: gradient is None!")
            all_passed = False
            continue

        max_diff = (g_orig - g_fused).abs().max().item()
        mean_diff = (g_orig - g_fused).abs().mean().item()
        passed = max_diff < 1e-3

        status = "PASS" if passed else "FAIL"
        print(f"  {name:8s}: max_diff={max_diff:.2e}, mean_diff={mean_diff:.2e} [{status}]")

        if not passed:
            all_passed = False

    print(f"\n  OVERALL: {'ALL PASSED' if all_passed else 'SOME FAILED'}")
    print()
    return all_passed


# =============================================================================
# MAIN
# =============================================================================
if __name__ == '__main__':
    assert torch.cuda.is_available(), "CUDA is required for this test"
    device = 'cuda'

    results = []
    results.append(("Isolated prime+conv", test_isolated_prime_conv(device)))
    results.append(("Full module e2e",     test_full_module(device)))
    results.append(("Multiple configs",    test_multiple_configs(device)))
    results.append(("Gradient flow",       test_gradient_flow(device)))

    print("=" * 70)
    print("SUMMARY")
    print("=" * 70)
    all_good = True
    for name, passed in results:
        status = "PASS" if passed else "FAIL"
        print(f"  {name:30s} [{status}]")
        if not passed:
            all_good = False

    print()
    if all_good:
        print("All tests passed! Fused kernel matches original implementation.")
    else:
        print("Some tests FAILED. Check output above for details.")

In [ ]:
"""
Speed Benchmark: Original PyTorch ConvNN Attention vs Fused CUDA Kernel

Measures forward and backward pass times across multiple configurations.
Uses proper CUDA timing (events + synchronization) and warmup iterations.
"""
import torch
import torch.nn as nn
import numpy as np
import time
import csv
import os


# =============================================================================
# ORIGINAL IMPLEMENTATION
# =============================================================================
class MultiHeadConvNNAttention(nn.Module):
    def __init__(self, d_hidden, num_heads, attention_dropout, K,
                 convolution_type='depthwise', seq_length=197):
        super().__init__()
        assert d_hidden % num_heads == 0

        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.d_k = d_hidden // num_heads
        self.K = K
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        self.conv = nn.Conv1d(
            in_channels=self.d_k, out_channels=self.d_k,
            kernel_size=K, stride=K, padding=0,
            groups=self.d_k, bias=False
        )
        self.conv.weight.data.fill_(1.0)

    def split_head(self, x):
        B, S, _ = x.size()
        return x.view(B, S, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        B, _, S, _ = x.size()
        return x.transpose(1, 2).contiguous().view(B, S, self.d_hidden)

    def _prime(self, v, qk, K):
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K, dim=2, largest=True)
        topk_values = torch.softmax(topk_values, dim=-1)
        topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)
        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime

    def forward(self, x):
        B = x.shape[0]
        q = self.split_head(self.W_q(x))
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.seq_length)

        prime = self._prime(v_merged, am_merged, self.K)
        out = self.conv(prime)

        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out)
        return self.W_o(self.combine_heads(out))


# =============================================================================
# FUSED IMPLEMENTATION
# =============================================================================
class MultiHeadConvNNAttentionFused(nn.Module):
    def __init__(self, d_hidden, num_heads, attention_dropout, K, seq_length=197):
        super().__init__()
        assert d_hidden % num_heads == 0

        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.d_k = d_hidden // num_heads
        self.K = K
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        from convnn_fused import FusedPrimeConv
        self.fused_prime_conv = FusedPrimeConv(d_k=self.d_k, K=self.K)

    def split_head(self, x):
        B, S, _ = x.size()
        return x.view(B, S, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        B, _, S, _ = x.size()
        return x.transpose(1, 2).contiguous().view(B, S, self.d_hidden)

    def forward(self, x):
        B = x.shape[0]
        q = self.split_head(self.W_q(x))
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.seq_length)

        out = self.fused_prime_conv(v_merged, am_merged)

        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out)
        return self.W_o(self.combine_heads(out))


# =============================================================================
# WEIGHT SYNC
# =============================================================================
def sync_weights(original, fused):
    with torch.no_grad():
        fused.W_q.weight.data.copy_(original.W_q.weight.data)
        fused.W_k.weight.data.copy_(original.W_k.weight.data)
        fused.W_v.weight.data.copy_(original.W_v.weight.data)
        fused.W_o.weight.data.copy_(original.W_o.weight.data)
        fused.fused_prime_conv.conv_weight.data.copy_(
            original.conv.weight.data.squeeze(1)
        )


# =============================================================================
# CUDA-ACCURATE TIMING UTILITY
# =============================================================================
def benchmark_fn(fn, warmup=20, repeats=100):
    """
    Time a function using CUDA events for accurate GPU timing.

    Returns:
        dict with 'mean_ms', 'std_ms', 'min_ms', 'max_ms', 'median_ms'
    """
    # Warmup: let CUDA JIT, caching, etc. stabilize
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    # Timed runs using CUDA events (not wall-clock time)
    timings = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

        start.record()
        fn()
        end.record()

        torch.cuda.synchronize()
        timings.append(start.elapsed_time(end))  # milliseconds

    timings = np.array(timings)
    return {
        'mean_ms': timings.mean(),
        'std_ms': timings.std(),
        'min_ms': timings.min(),
        'max_ms': timings.max(),
        'median_ms': np.median(timings),
    }


# =============================================================================
# BENCHMARK SUITE
# =============================================================================
def run_benchmark(config, device='cuda', warmup=20, repeats=100):
    """
    Benchmark a single configuration. Returns timing results for
    forward-only and forward+backward for both implementations.
    """
    torch.manual_seed(0)

    B = config['batch']
    d_hidden = config['d_hidden']
    num_heads = config['heads']
    K = config['K']
    SL = config['sl']

    # Build models
    original = MultiHeadConvNNAttention(
        d_hidden=d_hidden, num_heads=num_heads,
        attention_dropout=0.0, K=K, seq_length=SL
    ).to(device)

    fused = MultiHeadConvNNAttentionFused(
        d_hidden=d_hidden, num_heads=num_heads,
        attention_dropout=0.0, K=K, seq_length=SL
    ).to(device)

    sync_weights(original, fused)

    original.train()
    fused.train()

    x = torch.randn(B, SL, d_hidden, device=device)

    # --- Forward-only benchmarks ---
    def fwd_original():
        with torch.no_grad():
            original(x)

    def fwd_fused():
        with torch.no_grad():
            fused(x)

    t_fwd_orig = benchmark_fn(fwd_original, warmup=warmup, repeats=repeats)
    t_fwd_fused = benchmark_fn(fwd_fused, warmup=warmup, repeats=repeats)

    # --- Forward + Backward benchmarks ---
    def fwd_bwd_original():
        # Zero grads
        original.zero_grad(set_to_none=True)
        out = original(x)
        loss = out.sum()
        loss.backward()

    def fwd_bwd_fused():
        fused.zero_grad(set_to_none=True)
        out = fused(x)
        loss = out.sum()
        loss.backward()

    t_fwdbwd_orig = benchmark_fn(fwd_bwd_original, warmup=warmup, repeats=repeats)
    t_fwdbwd_fused = benchmark_fn(fwd_bwd_fused, warmup=warmup, repeats=repeats)

    return {
        'config': config,
        'fwd_original': t_fwd_orig,
        'fwd_fused': t_fwd_fused,
        'fwdbwd_original': t_fwdbwd_orig,
        'fwdbwd_fused': t_fwdbwd_fused,
    }


def format_timing(t):
    """Format timing dict as 'mean ± std ms'."""
    return f"{t['mean_ms']:8.3f} ± {t['std_ms']:.3f} ms"


def print_results(all_results):
    """Pretty-print benchmark results as a table."""

    # Header
    print()
    print("=" * 130)
    print(f"{'Config':>40s}  |  {'Forward (ms)':^45s}  |  {'Forward+Backward (ms)':^45s}")
    print(f"{'':>40s}  |  {'Original':>20s}  {'Fused':>20s}  |  {'Original':>20s}  {'Fused':>20s}")
    print("-" * 130)

    for r in all_results:
        cfg = r['config']
        label = f"B={cfg['batch']}, d={cfg['d_hidden']}, H={cfg['heads']}, K={cfg['K']}, SL={cfg['sl']}"

        fo = r['fwd_original']
        ff = r['fwd_fused']
        bo = r['fwdbwd_original']
        bf = r['fwdbwd_fused']

        print(f"{label:>40s}  |  {format_timing(fo):>20s}  {format_timing(ff):>20s}  "
              f"|  {format_timing(bo):>20s}  {format_timing(bf):>20s}")

    # Speedup summary
    print()
    print("=" * 130)
    print(f"{'Config':>40s}  |  {'Fwd Speedup':>12s}  {'Fwd+Bwd Speedup':>16s}  "
          f"{'Fwd ΔGPU mem':>14s}  {'Fwd+Bwd ΔGPU mem':>18s}")
    print("-" * 130)

    for r in all_results:
        cfg = r['config']
        label = f"B={cfg['batch']}, d={cfg['d_hidden']}, H={cfg['heads']}, K={cfg['K']}, SL={cfg['sl']}"

        fwd_speedup = r['fwd_original']['mean_ms'] / r['fwd_fused']['mean_ms']
        bwd_speedup = r['fwdbwd_original']['mean_ms'] / r['fwdbwd_fused']['mean_ms']

        print(f"{label:>40s}  |  {fwd_speedup:>11.2f}x  {bwd_speedup:>15.2f}x")

    print("=" * 130)


def run_memory_comparison(config, device='cuda'):
    """Measure peak GPU memory for both implementations."""
    torch.manual_seed(0)

    B = config['batch']
    d_hidden = config['d_hidden']
    num_heads = config['heads']
    K = config['K']
    SL = config['sl']

    results = {}

    for name, ModelClass in [('original', MultiHeadConvNNAttention),
                              ('fused', MultiHeadConvNNAttentionFused)]:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        if name == 'original':
            model = ModelClass(
                d_hidden=d_hidden, num_heads=num_heads,
                attention_dropout=0.0, K=K, seq_length=SL
            ).to(device)
        else:
            model = ModelClass(
                d_hidden=d_hidden, num_heads=num_heads,
                attention_dropout=0.0, K=K, seq_length=SL
            ).to(device)

        x = torch.randn(B, SL, d_hidden, device=device)

        # Forward + backward to capture peak memory
        torch.cuda.reset_peak_memory_stats()
        model.zero_grad(set_to_none=True)
        out = model(x)
        loss = out.sum()
        loss.backward()
        torch.cuda.synchronize()

        results[name] = {
            'peak_mb': torch.cuda.max_memory_allocated() / (1024 ** 2),
            'current_mb': torch.cuda.memory_allocated() / (1024 ** 2),
        }

        del model, x, out, loss
        torch.cuda.empty_cache()

    return results


def save_csv(all_results, filepath='benchmark_results.csv'):
    """Save benchmark results to CSV for plotting."""
    rows = []
    for r in all_results:
        cfg = r['config']
        row = {
            'batch': cfg['batch'],
            'd_hidden': cfg['d_hidden'],
            'heads': cfg['heads'],
            'K': cfg['K'],
            'seq_length': cfg['sl'],
            'fwd_orig_mean_ms': r['fwd_original']['mean_ms'],
            'fwd_orig_std_ms': r['fwd_original']['std_ms'],
            'fwd_fused_mean_ms': r['fwd_fused']['mean_ms'],
            'fwd_fused_std_ms': r['fwd_fused']['std_ms'],
            'fwdbwd_orig_mean_ms': r['fwdbwd_original']['mean_ms'],
            'fwdbwd_orig_std_ms': r['fwdbwd_original']['std_ms'],
            'fwdbwd_fused_mean_ms': r['fwdbwd_fused']['mean_ms'],
            'fwdbwd_fused_std_ms': r['fwdbwd_fused']['std_ms'],
            'fwd_speedup': r['fwd_original']['mean_ms'] / r['fwd_fused']['mean_ms'],
            'fwdbwd_speedup': r['fwdbwd_original']['mean_ms'] / r['fwdbwd_fused']['mean_ms'],
        }
        rows.append(row)

    with open(filepath, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nResults saved to {filepath}")


# =============================================================================
# MAIN
# =============================================================================
if __name__ == '__main__':
    assert torch.cuda.is_available(), "CUDA required"
    device = 'cuda'

    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA: {torch.version.cuda}")
    print()

    # ---- Configurations ----
    # Covers your actual ViT-Base setup + scaling experiments
    configs = [
        # Small (fast sanity check)
        {'batch': 4,  'd_hidden': 192,  'heads': 3,   'K': 8,   'sl': 50,  'label': 'Small'},
        # ViT-Base (your primary setup)
        {'batch': 4,  'd_hidden': 768,  'heads': 12,  'K': 8,   'sl': 197, 'label': 'ViT-B (B=4)'},
        {'batch': 16, 'd_hidden': 768,  'heads': 12,  'K': 8,   'sl': 197, 'label': 'ViT-B (B=16)'},
        {'batch': 64, 'd_hidden': 768,  'heads': 12,  'K': 8,   'sl': 197, 'label': 'ViT-B (B=64)'},
        # Vary K (nearest neighbors)
        {'batch': 16, 'd_hidden': 768,  'heads': 12,  'K': 4,   'sl': 197, 'label': 'ViT-B K=4'},
        {'batch': 16, 'd_hidden': 768,  'heads': 12,  'K': 16,  'sl': 197, 'label': 'ViT-B K=16'},
        {'batch': 16, 'd_hidden': 768,  'heads': 12,  'K': 32,  'sl': 197, 'label': 'ViT-B K=32'},
        {'batch': 16, 'd_hidden': 768,  'heads': 12,  'K': 64,  'sl': 197, 'label': 'ViT-B K=64'},
        # ViT-Large
        {'batch': 4,  'd_hidden': 1024, 'heads': 16,  'K': 8,   'sl': 197, 'label': 'ViT-L (B=4)'},
        {'batch': 16, 'd_hidden': 1024, 'heads': 16,  'K': 8,   'sl': 197, 'label': 'ViT-L (B=16)'},
        # Dense prediction (longer sequences for detection/segmentation)
        {'batch': 4,  'd_hidden': 768,  'heads': 12,  'K': 8,   'sl': 576, 'label': 'ViT-B SL=576'},
        {'batch': 4,  'd_hidden': 768,  'heads': 12,  'K': 8,   'sl': 1024,'label': 'ViT-B SL=1024'},
    ]

    # ---- Run benchmarks ----
    print("Running speed benchmarks (this may take a few minutes)...")
    print()

    all_results = []
    for i, cfg in enumerate(configs):
        label = cfg.pop('label', '')
        print(f"[{i+1}/{len(configs)}] {label}: B={cfg['batch']}, d={cfg['d_hidden']}, "
              f"H={cfg['heads']}, K={cfg['K']}, SL={cfg['sl']}...", flush=True)
        try:
            result = run_benchmark(cfg, device=device, warmup=20, repeats=100)
            all_results.append(result)

            # Quick inline preview
            fwd_sp = result['fwd_original']['mean_ms'] / result['fwd_fused']['mean_ms']
            bwd_sp = result['fwdbwd_original']['mean_ms'] / result['fwdbwd_fused']['mean_ms']
            print(f"         -> Fwd: {fwd_sp:.2f}x, Fwd+Bwd: {bwd_sp:.2f}x")
        except RuntimeError as e:
            print(f"         -> SKIPPED (OOM or error): {e}")

        # Clear between configs
        torch.cuda.empty_cache()

    # ---- Print results ----
    print_results(all_results)

    # ---- Memory comparison (ViT-Base default) ----
    print("\n\nMemory Usage Comparison (ViT-Base, B=16, K=8):")
    print("-" * 50)
    mem_cfg = {'batch': 16, 'd_hidden': 768, 'heads': 12, 'K': 8, 'sl': 197}
    mem = run_memory_comparison(mem_cfg, device=device)
    for name, stats in mem.items():
        print(f"  {name:10s}: peak={stats['peak_mb']:.1f} MB")
    if mem['original']['peak_mb'] > 0:
        savings = (1 - mem['fused']['peak_mb'] / mem['original']['peak_mb']) * 100
        print(f"  Memory savings: {savings:.1f}%")

    # ---- Save CSV ----
    save_csv(all_results, 'benchmark_results.csv')